# 4. Octane correlations and the tetramethylbutane audit (Section 6.1, Table 4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singati2/LOYOLA-PAPER/blob/main/notebooks/04_octane_correlations.ipynb)

Recomputes all 60 correlations of Table 4 from the data and compares each with the value printed in `main.tex`.

Every check prints the value in the paper next to the recomputed value. The last cell fails if any of them differ.

In [1]:
# Setup: runs inside the repository, or on Google Colab (clones the repository first).
import os, sys, subprocess
if os.path.exists("../verify.py"):
    os.chdir("..")
elif not os.path.exists("verify.py"):
    if not os.path.exists("LOYOLA-PAPER"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Singati2/LOYOLA-PAPER.git"], check=True)
    os.chdir("LOYOLA-PAPER")
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "pandas", "scikit-learn"], check=True)
ROOT = os.getcwd()
for p in (ROOT, os.path.join(ROOT, "external"), os.path.join(ROOT, "external", "bp"), os.path.join(ROOT, "structural")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("working directory:", ROOT)

working directory: /Users/ganeshshiwakoti/LOYOLA-PAPER


In [2]:
import re, csv, time
import numpy as np, pandas as pd
RESULTS = []

def check(what, paper, got, tol=0.0):
    """Record one comparison between a value printed in the paper and the recomputed value."""
    if isinstance(paper, (int, float)) and not isinstance(paper, bool):
        ok = abs(float(paper) - float(got)) <= tol + 1e-12
    else:
        ok = paper == got
    RESULTS.append({"check": what, "paper": paper, "recomputed": got, "status": "OK" if ok else "MISMATCH"})
    print(f"{'OK      ' if ok else 'MISMATCH'} {what}: paper {paper}, recomputed {got}")
    return ok

def run(cmd, tail=25):
    """Run a repository script, show the end of its output and require exit status 0."""
    t = time.time()
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}   # Jupyter's inline backend must not leak into scripts
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    out = (r.stdout + r.stderr).strip().splitlines()
    print("\n".join(out[-tail:]))
    print(f"[exit {r.returncode}, {time.time() - t:.0f} s]")
    RESULTS.append({"check": f"script: {cmd}", "paper": "exit 0", "recomputed": f"exit {r.returncode}",
                    "status": "OK" if r.returncode == 0 else "MISMATCH"})
    return r.stdout

In [3]:
import octane_data as od
P = [od.alkane_pairs(r[0]) for r in od.OCTANES]
Y = {k: np.array([r[i + 1] for r in od.OCTANES]) for i, k in enumerate(od.PROPS)}
rows = [("$M_1", (0, 1, 0)), ("$M_2", (1, 0, 0)), ("$HM", (0, 2, 0)), ("${}^{m}", (-1, 0, 0)), ("$R ", (-0.5, 0, 0)),
        ("$\\chi", (0, -0.5, 0)), ("$H/2", (0, -1, 0)), ("$ISI", (1, -1, 0)), ("$GA", (0.5, -1, 0)), ("$AG", (-0.5, 1, 0)),
        ("$LO(0,0,1)$ &", (0, 0, 1)), ("$LO(0,0,2)$ &", (0, 0, 2))]
tex = open("main.tex").read()
tab = tex[tex.index(r"\label{tab:lo_octane}"):]
tab = tab[tab.index(r"\midrule"):tab.index(r"\bottomrule")]
lines = [l for l in tab.splitlines() if "&" in l]
for (prefix, t), line in zip(rows, lines):
    printed = [float(x) for x in re.findall(r"([+-]?\d\.\d{3})", line)]
    x = np.array([od.lo_pairs(p, *t) for p in P])
    for k, pv in zip(od.PROPS, printed):
        r = np.corrcoef(x, Y[k])[0, 1]
        check(f"r({line.split('&')[0].strip()[:22]}, {k})", pv, round(r, 3), 0.0006)

OK       r($M_1 = LO(0,1,0)$, T_B): paper -0.718, recomputed -0.718
OK       r($M_1 = LO(0,1,0)$, dHf): paper -0.762, recomputed -0.762
OK       r($M_1 = LO(0,1,0)$, dHvap): paper -0.935, recomputed -0.935
OK       r($M_1 = LO(0,1,0)$, S): paper -0.97, recomputed -0.97
OK       r($M_1 = LO(0,1,0)$, omega): paper -0.97, recomputed -0.97
OK       r($M_2 = LO(1,0,0)$, T_B): paper -0.497, recomputed -0.497
OK       r($M_2 = LO(1,0,0)$, dHf): paper -0.542, recomputed -0.542
OK       r($M_2 = LO(1,0,0)$, dHvap): paper -0.811, recomputed -0.811
OK       r($M_2 = LO(1,0,0)$, S): paper -0.934, recomputed -0.934
OK       r($M_2 = LO(1,0,0)$, omega): paper -0.988, recomputed -0.988
OK       r($HM = LO(0,2,0)$, T_B): paper -0.654, recomputed -0.654
OK       r($HM = LO(0,2,0)$, dHf): paper -0.71, recomputed -0.71
OK       r($HM = LO(0,2,0)$, dHvap): paper -0.903, recomputed -0.903
OK       r($HM = LO(0,2,0)$, S): paper -0.971, recomputed -0.971
OK       r($HM = LO(0,2,0)$, omega): paper -0.982, rec

### The tetramethylbutane audit (`TMB_DHVAP_AUDIT.md`)

Four values are reported separately; none replaces the benchmark value.

In [4]:
run("python tmb_dhvap_audit.py", tail=6)
aud = pd.read_csv("tmb_dhvap_audit.csv").set_index("variant")
print(aud[["tmb_dhvap_kcal", "top_index", "top_abs_r", "abs_r_LO001"]])
for name, want in (("original", 0.984), ("excluded", 0.980), ("hypothetical liquid estimate", 0.976), ("NIST 298 K, solid to gas", 0.374)):
    check(f"|r| of LO(0,0,1) on dHvap, {name}", want, round(aud.loc[name, "abs_r_LO001"], 3), 0.0006)
    check(f"column leader, {name}", "LO(0,0,1)", aud.loc[name, "top_index"])

['original', '8.410', 'LO(0,0,1)', '0.9838', 'LO(0,0,2)', '0.9813', '0.9838', '0.3827']
['excluded', '', 'LO(0,0,1)', '0.9799', 'LO(0,0,2)', '0.9795', '0.9799', '0.2794']
['hypothetical liquid estimate', '8.564', 'LO(0,0,1)', '0.9761', 'LO(0,0,2)', '0.9696', '0.9761', '0.3675']
['NIST 298 K, solid to gas', '10.263', 'LO(0,0,1)', '0.3736', 'LO(0,0,2)', '0.3317', '0.3736', '0.0183']
[exit 0, 0 s]
                              tmb_dhvap_kcal  top_index  top_abs_r  \
variant                                                              
original                               8.410  LO(0,0,1)     0.9838   
excluded                                 NaN  LO(0,0,1)     0.9799   
hypothetical liquid estimate           8.564  LO(0,0,1)     0.9761   
NIST 298 K, solid to gas              10.263  LO(0,0,1)     0.3736   

                              abs_r_LO001  
variant                                    
original                           0.9838  
excluded                           0.9799  
hypot

## Summary

In [5]:
summary = pd.DataFrame(RESULTS)
display(summary) if "display" in dir() else print(summary.to_string())
n_bad = int((summary.status != "OK").sum())
print(f"{len(summary)} checks, {n_bad} mismatches")
assert n_bad == 0, "some recomputed values differ from the paper"

                                                      check      paper recomputed status
0                                 r($M_1 = LO(0,1,0)$, T_B)     -0.718     -0.718     OK
1                                 r($M_1 = LO(0,1,0)$, dHf)     -0.762     -0.762     OK
2                               r($M_1 = LO(0,1,0)$, dHvap)     -0.935     -0.935     OK
3                                   r($M_1 = LO(0,1,0)$, S)      -0.97      -0.97     OK
4                               r($M_1 = LO(0,1,0)$, omega)      -0.97      -0.97     OK
5                                 r($M_2 = LO(1,0,0)$, T_B)     -0.497     -0.497     OK
6                                 r($M_2 = LO(1,0,0)$, dHf)     -0.542     -0.542     OK
7                               r($M_2 = LO(1,0,0)$, dHvap)     -0.811     -0.811     OK
8                                   r($M_2 = LO(1,0,0)$, S)     -0.934     -0.934     OK
9                               r($M_2 = LO(1,0,0)$, omega)     -0.988     -0.988     OK
10                   